# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## Preparing the data

### Transforming the csv data to a numpy array

In [390]:
import numpy as np

usdYen_raw_data = np.genfromtxt("./data/currency-data/USD-JPY.csv", skip_header=1, delimiter=";", usecols=1)

print("Length: ",len(usdYen_raw_data))
print("Data type: ",usdYen_raw_data.dtype)
print("Raw Data: ",usdYen_raw_data)

Length:  550
Data type:  float64
Raw Data:  [155.21  155.81  155.34  156.15  156.39  154.54  153.4   154.    152.85
 150.62  151.15  147.44  149.49  147.94  147.66  147.38  147.02  146.93
 147.18  147.72  147.36  147.66  148.81  147.4   144.55  144.65  146.07
 144.09  144.85  144.021 142.55  145.581 145.323 144.918 143.623 142.15
 143.49  146.935 149.83  149.275 148.61  148.018 150.545 149.262 152.254
 151.377 155.182 155.92  156.266 157.657 157.273 157.772 156.407 153.683
 149.989 149.743 154.73  154.309 152.608 152.946 152.244 149.485 149.09
 148.679 142.15  143.808 140.764 142.251 146.176 144.367 147.579 146.634
 146.519 153.718 157.44  157.848 160.719 160.835 159.767 157.323 156.752
 157.29  156.943 155.619 155.703 152.949 158.296 154.562 153.24  151.609
 151.303 151.429 149.033 147.03  150.064 150.473 150.181 149.256 148.308
 148.149 148.128 144.904 144.603 140.998 142.401 142.141 144.893 146.772
 149.403 149.562 151.472 149.333 149.599 149.831 149.515 149.274 149.33
 148.36  147.

As the currency data is from newer to older, the order should be inverted.

In [391]:
usdYen_raw_data = np.flip(usdYen_raw_data, axis=0)
print(usdYen_raw_data)

[123.408 122.681 123.861 122.851 122.83  124.05  123.79  123.85  124.197
 124.276 122.002 121.703 119.017 120.567 119.92  120.584 119.898 120.246
 119.442 121.452 120.608 123.136 122.579 122.755 122.741 123.056 120.954
 121.146 120.289 120.203 117.234 116.98  118.71  121.14  116.808 113.236
 112.603 113.975 113.735 113.96  111.543 113.049 111.64  108.079 108.762
 111.793 106.282 107.094 108.633 110.097 110.214 106.532 106.91  104.11
 102.19  102.518 100.5   104.651 106.149 102.047 101.821 101.25  100.21
 101.81  103.975 102.67  102.27  101.05  101.32  102.893 104.146 103.8
 104.694 103.089 106.635 110.874 113.229 113.503 115.274 117.84  117.32
 116.95  117.02  114.467 114.6   115.08  112.584 113.205 112.836 112.134
 114.005 114.77  112.69  111.329 111.383 111.044 108.48  109.08  111.495
 112.7   113.28  111.16  111.31  110.357 110.327 110.67  111.26  112.36
 113.869 112.48  111.153 110.654 110.64  109.05  109.14  109.36  110.247
 107.83  110.731 111.958 112.428 112.545 111.791 113.479 

### Computing the numer of samples for each data split

In [392]:
train_samples_number = len(usdYen_raw_data)
print("Number of train samples: ", train_samples_number)

Number of train samples:  550


### Normalizing data

In [393]:
'''
mean = usdYen_raw_data[:train_samples_number].mean(axis=0)
usdYen_raw_data -= mean
std = usdYen_raw_data[:train_samples_number].std(axis=0)
usdYen_raw_data /= std

print(usdYen_raw_data)
'''

'\nmean = usdYen_raw_data[:train_samples_number].mean(axis=0)\nusdYen_raw_data -= mean\nstd = usdYen_raw_data[:train_samples_number].std(axis=0)\nusdYen_raw_data /= std\n\nprint(usdYen_raw_data)\n'

### Creating timeseries data

In [398]:
from tensorflow import keras

# Parameters
sampling_rate = 1
sequence_length = 4 # Observations will go back 4 weeks
delay = sampling_rate * (sequence_length + 4 - 1) # target is 4 weeks after the end of the sequence
batch_size = 64 # small batch size to start

# train dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    usdYen_raw_data[:-delay],
    targets=usdYen_raw_data[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

Checking that it works correctly

In [399]:
for inputs, targets in train_dataset:
    for i in range(inputs.shape[0]):
        print([float(x) for x in inputs[i]], float(targets[i]))

[123.408, 122.681, 123.861, 122.851] 123.85
[122.681, 123.861, 122.851, 122.83] 124.197
[123.861, 122.851, 122.83, 124.05] 124.276
[122.851, 122.83, 124.05, 123.79] 122.002
[122.83, 124.05, 123.79, 123.85] 121.703
[124.05, 123.79, 123.85, 124.197] 119.017
[123.79, 123.85, 124.197, 124.276] 120.567
[123.85, 124.197, 124.276, 122.002] 119.92
[124.197, 124.276, 122.002, 121.703] 120.584
[124.276, 122.002, 121.703, 119.017] 119.898
[122.002, 121.703, 119.017, 120.567] 120.246
[121.703, 119.017, 120.567, 119.92] 119.442
[119.017, 120.567, 119.92, 120.584] 121.452
[120.567, 119.92, 120.584, 119.898] 120.608
[119.92, 120.584, 119.898, 120.246] 123.136
[120.584, 119.898, 120.246, 119.442] 122.579
[119.898, 120.246, 119.442, 121.452] 122.755
[120.246, 119.442, 121.452, 120.608] 122.741
[119.442, 121.452, 120.608, 123.136] 123.056
[121.452, 120.608, 123.136, 122.579] 120.954
[120.608, 123.136, 122.579, 122.755] 121.146
[123.136, 122.579, 122.755, 122.741] 120.289
[122.579, 122.755, 122.741, 123.

Inspecting the output

In [400]:
for samples, targets in train_dataset:
    print("Samples: ", samples)
    print("Sample shape: ", samples.shape)
    print("Targets: ", targets)
    print("Targets shape: ", targets.shape)
    break

Samples:  tf.Tensor(
[[123.408 122.681 123.861 122.851]
 [122.681 123.861 122.851 122.83 ]
 [123.861 122.851 122.83  124.05 ]
 [122.851 122.83  124.05  123.79 ]
 [122.83  124.05  123.79  123.85 ]
 [124.05  123.79  123.85  124.197]
 [123.79  123.85  124.197 124.276]
 [123.85  124.197 124.276 122.002]
 [124.197 124.276 122.002 121.703]
 [124.276 122.002 121.703 119.017]
 [122.002 121.703 119.017 120.567]
 [121.703 119.017 120.567 119.92 ]
 [119.017 120.567 119.92  120.584]
 [120.567 119.92  120.584 119.898]
 [119.92  120.584 119.898 120.246]
 [120.584 119.898 120.246 119.442]
 [119.898 120.246 119.442 121.452]
 [120.246 119.442 121.452 120.608]
 [119.442 121.452 120.608 123.136]
 [121.452 120.608 123.136 122.579]
 [120.608 123.136 122.579 122.755]
 [123.136 122.579 122.755 122.741]
 [122.579 122.755 122.741 123.056]
 [122.755 122.741 123.056 120.954]
 [122.741 123.056 120.954 121.146]
 [123.056 120.954 121.146 120.289]
 [120.954 121.146 120.289 120.203]
 [121.146 120.289 120.203 117.234]

## Simple LSTM Model

In [ ]:
from keras import models
from keras import layers

def build_lstm_model():
    model = models.Sequential()
    model.add(layers.LSTM(16, input_shape=(sequence_length, 1)))
    model.add(layers.Dense(1))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

'''
model = build_lstm_model()
callbacks = [keras.callbacks.ModelCheckpoint("usd_jpy_lstm.keras", save_best_only=True)]
history = model.fit(train_dataset, epochs=20, validation_data=validation_dataset, callbacks=callbacks)
model = keras.models.load_model("usd_jpy_lstm.keras")
model.summary()
print(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")
'''
    

'\nfrom keras import models\nfrom keras import layers\n\ndef build_lstm_model():\n    model = models.Sequential()\n    model.add(layers.LSTM(16, input_shape=(sequence_length, 1)))\n    model.add(layers.Dense(1))\n    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])\n    return model\n\nmodel = build_lstm_model()\ncallbacks = [keras.callbacks.ModelCheckpoint("usd_jpy_lstm.keras", save_best_only=True)]\nhistory = model.fit(train_dataset, epochs=20, validation_data=validation_dataset, callbacks=callbacks)\nmodel = keras.models.load_model("usd_jpy_lstm.keras")\nmodel.summary()\nprint(f"Test MAE: {model.evaluate(test_dataset)[1]:.2f}")\n'